# 04 — Momentum Baseline Inference

Formal inference for the already-validated momentum baseline.

**Pre-specified convention:** Newey–West/HAC lag 6 months as primary; 12 months as robustness.

This notebook still does **not** condition on market concentration.


In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import monthly_rank_ic, quintile_returns, quintile_return_panel
from src.inference import primary_and_robustness_hac


## 1. Rebuild baseline series


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
clean, _ = build_clean_monthly_panel(RAW, CleaningConfig())
history = build_monthly_history_panel(RAW)
mom = attach_momentum_to_formation_universe(clean, history)
mom = assign_signal_quintiles(mom)

rank_ic = monthly_rank_ic(mom)
qret = quintile_returns(mom)
vw = quintile_return_panel(qret, return_col='VW_Return')
ew = quintile_return_panel(qret, return_col='EW_Return')


## 2. Rank IC inference


In [ ]:
rank_ic_hac = primary_and_robustness_hac(rank_ic['RankIC'])
rank_ic_hac


## 3. Value-weighted Q5−Q1 inference (primary portfolio)


In [ ]:
vw_hac = primary_and_robustness_hac(vw['Q5_minus_Q1'])
vw_hac


## 4. Equal-weighted Q5−Q1 inference (robustness)


In [ ]:
ew_hac = primary_and_robustness_hac(ew['Q5_minus_Q1'])
ew_hac


## Interpretation rule

Interpret effect size, sign, uncertainty, and robustness together. Do not use a p-value alone as the conclusion, and do not change the HAC lag because one specification crosses a conventional significance threshold.
